In [ ]:
import sys

from hydra.utils import instantiate
from einops import rearrange
from omegaconf import OmegaConf
import torch
import numpy as np

In [ ]:
snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/32_ae-murp-ft/2025-08-19/02-50-09/0"
config_path = f"{snapshot_dir}/.hydra"
print(config_path)

cfg = OmegaConf.load(config_path + "/config.yaml")

In [ ]:
# Register custom resolver to handle multiplication in OmegaConf interpolation
OmegaConf.register_new_resolver("mul", lambda x, y: float(x) * float(y))
OmegaConf.register_new_resolver("div", lambda a, b: float(a) / float(b))

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models"))

model = instantiate(cfg.model)
model = model.cuda().eval()
snapshot = torch.load(f"{snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
model.load_state_dict(snapshot["model"])
val_dataloader = instantiate(cfg.val_data_loader)

In [ ]:
bs_snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/32_ae-murp-ft/2025-08-19/02-50-09/0"
bs_config_path = f"{bs_snapshot_dir}/.hydra"
print(bs_config_path)

bs_cfg = OmegaConf.load(bs_config_path + "/config.yaml")

In [ ]:
# bs_snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/gt_droid_egodex/2025-07-28/23-50-47/0"
another_snapshot_dir = "/home/ravenhuang/h2r/robot_world_models/projects/latent_action_models/data/experiments/gt_pr-murp/2025-08-16/01-43-46/0"
another_config_path = f"{another_snapshot_dir}/.hydra"
print(another_config_path)

another_cfg = OmegaConf.load(another_config_path + "/config.yaml")

In [ ]:
bs_model = instantiate(bs_cfg.model)
bs_model = bs_model.cuda().eval()
snapshot = torch.load(f"{bs_snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
bs_model.load_state_dict(snapshot["model"])

In [ ]:
another_model = instantiate(another_cfg.model)
another_model = another_model.cuda().eval()
snapshot = torch.load(f"{another_snapshot_dir}/snapshot.pt", map_location="cuda:0", weights_only=True)
another_model.load_state_dict(snapshot["model"])

In [ ]:
bs_cfg.val_data_loader[0].dataset.datasets['Murp'].transform.sample_size = 8
bs_cfg.val_data_loader[0].batch_size = 2
bs_val_dataloader = instantiate(bs_cfg.val_data_loader)
murp_dataloader = bs_val_dataloader[0]

In [ ]:
for i, murp_batch in enumerate(murp_dataloader):
    if i < 2:
        continue
    if i > 3:
        break

In [ ]:
murp_batch["rgb"].shape

In [ ]:
bs_model.ae_only = True  # Set to True to use autoencoder only
z = bs_model.visualize( murp_batch["rgb"][1:2,i:i+8].cuda(),  murp_batch["actions"][1:2, i:i+8].cuda(),
                                morphology_index = murp_batch["morphology_index"][1:2].cuda(),
                                ee_action_dim = murp_batch["ee_action_dim"][1:2].cuda() ).detach().cpu().numpy()

In [ ]:
bs_val_dataloader[0].dataset.datasets.keys()

In [ ]:
cfg.val_data_loader[0].dataset.datasets['Droid'].transform.sample_size = 64
cfg.val_data_loader[1].dataset.datasets['EgoDex'].transform.sample_size = 64
cfg.val_data_loader[3].dataset.datasets['MPK'].transform.sample_size = 64
val_dataloader = instantiate(cfg.val_data_loader)
droid_dataloader = val_dataloader[0]
egodex_dataloader = val_dataloader[1]
mpk_dataloader = val_dataloader[3]

In [ ]:
droid_zs = []
for i, droid_batch in enumerate(droid_dataloader):
    rgb = droid_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x).detach().cpu().numpy()
    droid_zs.append(z)
    if i > 100:
        break

egodex_zs = []
for i, egodex_batch in enumerate(egodex_dataloader):
    rgb = egodex_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x).detach().cpu().numpy()
    egodex_zs.append(z)
    if i > 100:
        break
mpk_zs = []
for i, mpk_batch in enumerate(mpk_dataloader):
    rgb = mpk_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x).detach().cpu().numpy()
    mpk_zs.append(z)
    if i > 100:
        break

In [ ]:
for i, murp_batch in enumerate(murp_dataloader):
    if i < 2:
        continue
    if i > 3:
        break

In [ ]:
all_z = []
bs_model.ae_only = True  # Set to True to use autoencoder only
for i in range(0, 64, 8):
    z = bs_model.visualize( murp_batch["rgb"][1:2,i:i+8].cuda(),  murp_batch["actions"][1:2, i:i+8].cuda(),
                                    morphology_index = murp_batch["morphology_index"][1:2].cuda(),
                                   ee_action_dim = murp_batch["ee_action_dim"][1:2].cuda() ).detach().cpu().numpy()
    all_z.append(z)

In [ ]:
all_z = np.concatenate(np.array(all_z)[:,0], axis=0)
new_z = np.concatenate([all_z[:,:,:256], all_z[:,:,512:768]], axis=2)

In [ ]:
media.write_video(f"zmurp.mp4", rearrange(new_z, "t c h w -> t h w c") , fps=10)

In [ ]:
for i, egodex_batch in enumerate(egodex_dataloader):
    if i < 1:
        continue
    if i > 2:
        break

In [ ]:
all_z = []
model.ae_only = True  # Set to True to use autoencoder only
for i in range(0, 64, 8):
    z = model.visualize( egodex_batch["rgb"][1:2,i:i+8].cuda(),  egodex_batch["actions"][1:2, i:i+8].cuda(),
                                    morphology_index = egodex_batch["morphology_index"][1:2].cuda(),
                                   ee_action_dim = egodex_batch["ee_action_dim"][1:2].cuda() ).detach().cpu().numpy()
    all_z.append(z)

In [ ]:
all_z = np.concatenate(np.array(all_z)[:,0], axis=0)
new_z = np.concatenate([all_z[:,:,:256], all_z[:,:,512:768]], axis=2)

In [ ]:
for i, mpk_batch in enumerate(mpk_dataloader):
    if i < 2:
        continue
    if i > 3:
        break


In [ ]:
all_z = []
model.ae_only = True  # Set to True to use autoencoder only
for i in range(0, 64, 8):
    z = model.visualize( mpk_batch["rgb"][2:3,i:i+8].cuda(),  mpk_batch["actions"][2:3, i:i+8].cuda(),
                                    morphology_index = mpk_batch["morphology_index"][2:3].cuda(),
                                   ee_action_dim = mpk_batch["ee_action_dim"][2:3].cuda() ).detach().cpu().numpy()
    all_z.append(z)

In [ ]:
all_z = np.concatenate(np.array(all_z)[:,0], axis=0)
new_mpk_z = np.concatenate([all_z[:,:,:256], all_z[:,:,512:768]], axis=2)

In [ ]:
import mediapy as media
media.write_video(f"zmpk.mp4", rearrange(new_mpk_z[:30], "t c h w -> t h w c") , fps=10)

In [ ]:
murp_zs = []
murp_zs_another = []
bs_model.ae_only = True  # Set to True to use autoencoder only
for i, murp_batch in enumerate(murp_dataloader):
    if i < 0:
        continue
    rgb = murp_batch["rgb"].cuda()
    z = bs_model.visualize( rgb,  murp_batch["actions"].cuda(),
                                    morphology_index = murp_batch["morphology_index"].cuda(),
                                   ee_action_dim = murp_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    # z_another = another_model.visualize( rgb,  murp_batch["actions"].cuda(),
    #                                 morphology_index = murp_batch["morphology_index"].cuda(),
    #                                ee_action_dim = murp_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    murp_zs.append(z)
    # murp_zs_another.append(z_another)
    if i > 4:
        break


In [ ]:
murp_zs = np.array(murp_zs)
murp_zs = np.concatenate(murp_zs, axis=0)
murp_zs_another = np.array(murp_zs_another)
murp_zs_another = np.concatenate(murp_zs_another, axis=0)

In [ ]:
import mediapy as media
for i,z in enumerate(murp_zs):
    media.write_video(f"z{i}.mp4", rearrange(z, "t c h w -> t h w c") , fps=10)

In [ ]:
all_p = []
for start in range(0,64,8):
    predictions = bs_model._generate_future(rgb[1:2, start:start+8])
    all_p.extend(predictions.detach().cpu().numpy())

In [ ]:
predictions = bs_model._generate_future(rgb[1:2, start:start+8])

In [ ]:
import numpy as np
all_p = np.concatenate(np.array(all_p),axis=0)

In [ ]:
import mediapy as media
media.write_video("all_p.mp4", rearrange(all_p, "t c h w -> t h w c") , fps=10)

In [ ]:
droid_zs_bs = []
for i, droid_batch in enumerate(droid_dataloader):
    z = bs_model._forward_action_encode( droid_batch["actions"].cuda(),
                                    droid_batch["morphology_index"].cuda(),
                                    droid_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    droid_zs_bs.append(z)
    if i > 50:
        break
egodex_zs_bs = []
for i, egodex_batch in enumerate(egodex_dataloader):
    z = bs_model._forward_action_encode( egodex_batch["actions"].cuda(),
                                    egodex_batch["morphology_index"].cuda(),
                                    egodex_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    egodex_zs_bs.append(z)
    if i > 50:
        break

mpk_zs_bs = []
for i, mpk_batch in enumerate(mpk_dataloader):
    z = bs_model._forward_action_encode( mpk_batch["actions"].cuda(),
                                    mpk_batch["morphology_index"].cuda(),
                                    mpk_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    mpk_zs_bs.append(z)
    if i > 50:
        break

In [ ]:
mpk_batch['rgb'].shape

In [ ]:
import mediapy as media
media.write_video("droid_0.mp4", rearrange(droid_batch['rgb'][0].cpu().numpy(), "t c h w -> t h w c"), fps=10)

In [ ]:
hot3d_zs = []
for i, hot3d_batch in enumerate(hot3d_dataloader):
    rgb = hot3d_batch["rgb"].cuda()
    x = model._forward_tokenizer_encode(rgb=rgb)
    z = model._forward_inverse_model(x).detach().cpu().numpy()
    hot3d_zs.append(z)
    if i > 50:
        break


In [ ]:
hot3d_zs_bs = []
for i, hot3d_batch in enumerate(hot3d_dataloader):
    z = bs_model._forward_action_encode( hot3d_batch["actions"].cuda(),
                                    hot3d_batch["morphology_index"].cuda(),
                                    hot3d_batch["ee_action_dim"].cuda() ).detach().cpu().numpy()
    hot3d_zs_bs.append(z)
    if i > 50:
        break

In [ ]:
import numpy as np

In [ ]:
droid_zs_c = np.concatenate(droid_zs, axis=0)
egodex_zs_c = np.concatenate(egodex_zs, axis=0)
mpk_zs_c = np.concatenate(mpk_zs, axis=0)

In [ ]:
droid_zs_bs_c = np.concatenate(droid_zs_bs, axis=0)
egodex_zs_bs_c = np.concatenate(egodex_zs_bs, axis=0)
mpk_zs_bs_c = np.concatenate(mpk_zs_bs, axis=0)

In [ ]:
flatten_droid_zs = rearrange(droid_zs_c, "b t a m -> (b t) (a m)")
flatten_egodex_zs = rearrange(egodex_zs_c, "b t a m -> (b t) (a m)")
flatten_mpk_zs = rearrange(mpk_zs_c, "b t a m -> (b t) (a m)")

In [ ]:
flatten_droid_zs_bs = rearrange(droid_zs_bs_c, "b t a m -> (b t a) m")
flatten_egodex_zs_bs = rearrange(egodex_zs_bs_c, "b t a m -> (b t a) m")
flatten_mpk_zs_bs = rearrange(mpk_zs_bs_c, "b t a m -> (b t a) m")

In [ ]:
embeddings = np.stack([flatten_droid_zs, flatten_egodex_zs, flatten_mpk_zs], axis=0)
# embeddings = np.stack([flatten_droid_zs, flatten_egodex_zs], axis=0)


In [ ]:
bs_embeddings = np.stack([flatten_droid_zs_bs, flatten_egodex_zs_bs, flatten_mpk_zs_bs], axis=0)
# bs_embeddings = np.stack([flatten_droid_zs_bs, flatten_egodex_zs_bs], axis=0)


In [ ]:
import umap
import numpy as np
import plotly.express as px
import torch

def plot_umap_3d_interactive(embeddings, color_labels=None):
    """
    embeddings: torch.Tensor or np.ndarray of shape (S, B, M)
    color_labels: optional array of shape (S*B,) for coloring
    """
    # Convert to numpy
    if torch.is_tensor(embeddings):
        embeddings = embeddings.detach().cpu().numpy()

    S, B, M = embeddings.shape
    embeddings_flat = embeddings.reshape(S * B, M)

    # Default color labels = data source index
    if color_labels is None:
        color_labels = np.repeat(np.arange(S), B)

    # Run UMAP
    reducer = umap.UMAP(n_components=3, random_state=42)
    embeddings_3d = reducer.fit_transform(embeddings_flat)

    # Create interactive plot
    fig = px.scatter_3d(
        x=embeddings_3d[:, 0],
        y=embeddings_3d[:, 1],
        z=embeddings_3d[:, 2],
        color=color_labels.astype(str),  # plotly requires string or category
        labels={'color': 'Data Source'},
        title="Interactive 3D UMAP"
    )
    fig.update_traces(marker=dict(size=3, opacity=0.7))
    fig.update_layout(margin=dict(l=0, r=0, b=0, t=30))
    fig.show()


In [ ]:
S, B, M = embeddings.shape

In [ ]:
plot_umap_3d_interactive(embeddings[...,:M//2], color_labels=np.repeat(np.arange(3), embeddings.shape[1]))

In [ ]:
plot_umap_3d_interactive(bs_embeddings, color_labels=np.repeat(np.arange(3), bs_embeddings.shape[1]))


In [ ]:
embeddings.shape

In [ ]:
flatten_droid_zs.shape

In [ ]:
droid_internal = rearrange(droid_zs_c, "b t a m -> (b t) a m")

In [ ]:
droid_internal.shape

In [ ]:
plot_umap_3d_interactive(droid_internal[:10], color_labels=np.repeat(np.arange(10), droid_internal[:10].shape[1]))
